# **Exercise 3**: Build your own model for Charleston

In this exercise we will create our own SFINCS model for the area around Charleston. After doing this exercise you will know how to set up a basic SFINCS model from scratch, now using the yml-file option. 


## **Build a model from CLI**

In this example the Charleston SFINCS compound flood model will be made, using HydroMT’s yml-file that allows for fast model configuration. 

In [1]:
# To check the version of hydromt and the hydromt_sfincs plugin, run the following command in a terminal:
!hydromt --models

model plugins:
 - sfincs (hydromt_sfincs 1.0.3.dev0)
generic models (hydromt 0.9.2.dev):
 - grid_model
 - vector_model
 - network_model



Using the yml-file option, the model can be build from the command line interface (CLI) using the following command:

    hydromt build sfincs "./input" -i "./sfincs_build_manning.yml" -vv

For more information see the full online example notebook: https://deltares.github.io/hydromt_sfincs/latest/_examples/build_from_cli.html

In [2]:
# For more information on command line available options, type:
!hydromt build --help

Usage: hydromt build [OPTIONS] MODEL MODEL_ROOT

  Build models from scratch.

  Example usage: --------------

  To build a wflow model for a subbasin using a point coordinates snapped to
  cells with upstream area >= 50 km2 hydromt build wflow /path/to/model_root
  -i /path/to/wflow_config.ini  -r "{'subbasin': [-7.24, 62.09], 'uparea':
  50}" -d deltares_data -d /path/to/data_catalog.yml -v

  To build a sfincs model based on a bbox hydromt build sfincs
  /path/to/model_root  -i /path/to/sfincs_config.ini  -r "{'bbox':
  [4.6891,52.9750,4.9576,53.1994]}"  -d /path/to/data_catalog.yml -v

Options:
  --opt TEXT               Method specific keyword arguments, see the method
                           documentation of the specific model for more
                           information about the arguments.
  -i, --config PATH        Path to hydroMT configuration file, for the model
                           specific implementation.
  -r, --region TEXT        Set the region for which to 

In [5]:
!hydromt build sfincs "./sfincs_charleston_cli" --region "{'geom': '../local_data/region.geojson'}" -i "./BONUS_build_model_from_yml.yml" --force-overwrite -vv

2023-12-20 15:51:18,894 - build - log - DEBUG - Writing log messages to new file C:\Users\eilan_dk\work\sfincs_DSD23\02_hands-on\notebook_answers\sfincs_charleston_cli\hydromt.log.
2023-12-20 15:51:18,894 - build - log - INFO - HydroMT version: 0.9.2.dev
2023-12-20 15:51:18,894 - build - main - INFO - Building instance of sfincs model at C:\Users\eilan_dk\work\sfincs_DSD23\02_hands-on\notebook_answers\sfincs_charleston_cli.
2023-12-20 15:51:18,894 - build - main - INFO - User settings:
2023-12-20 15:51:18,988 - build - data_catalog - INFO - Parsing data catalog from C:\Users\eilan_dk\work\sfincs_DSD23\02_hands-on\notebook_answers\data_catalog.yml
2023-12-20 15:51:19,010 - build - model_api - INFO - Initializing sfincs model from hydromt_sfincs (v1.0.3.dev0).
2023-12-20 15:51:19,011 - build - model_api - DEBUG - Setting model config options.
2023-12-20 15:51:19,011 - build - model_api - INFO - setup_grid_from_region.region: {'geom': '../local_data/region.geojson'}
2023-12-20 15:51:19,01

**Explanation of what is provided:**
    
* `!` : the '!' is added so you can run the command line interface (CLI) from a python notebook
* `hydromt build sfincs` : HydroMT should build a SFINCS model
* `./sfincs_charleston_cli` : HydroMT should build the model in a folder called "./sfincs_charleston_cli" relative to the current working directory (you can also provide absolute paths)
* `--region \"{'geom': '../local_data/region.geojson'}\"` : the area of interest for which a model is created is based on a geometry, which is already defined for you in the file \"../local_data/region.geojson\ (same as in the other example, after determining the wanted watersheds)
* `-i BONUS_build_model_from_yml.yml` : model configuration which describes the complete pipeline to build your model, more on that later
* `--force-overwrite` :  even if there's already an existing folder with the same name and SFINCS input files, HydroMT will overwrite it
* `-v` : add verbosity to the logger

**Hint:** Make sure that the (relative) paths and data sources in your data_catalog are correct

## **Explanation of HydroMT’s .yml-file:**

In [4]:
fn = "BONUS_build_model_from_yml.yml"
with open(fn, "r") as f:
    txt = f.read()
print(txt)

global:
  data_libs: ["./data_catalog.yml", "deltares_data"]

setup_config:
  tref: 20161001 000000
  tstart: 20161007 000000
  tstop: 20161009 000000
  dtout: 3600
  dthisout: 600

setup_grid_from_region:
  res: 200                   
  crs: utm
  rotated: false
  
setup_dep:
  datasets_dep:
    - elevtn: topography_charleston  # update to your file
    - elevtn: gebco

setup_mask_active:
  mask: geojsons/region_bbox.geojson   # update to your file
  zmin: -5
  fill_area: 10
  include_mask: geojsons/include_mask.geojson  # update to your file
  exclude_mask: geojsons/exclude_mask.geojson  # update to your file

setup_mask_bounds:
  btype: waterlevel
  zmax: -2
  exclude_mask: geojsons/exclude_mask.geojson  # update to your file
  
setup_manning_roughness:
  rgh_lev_land: 0
  manning_land: 0.06
  manning_sea : 0.02

setup_cn_infiltration:
  cn: gcn250
  antecedent_moisture: avg

setup_waterlevel_forcing:
  geodataset: gtsm_codec_reanalysis_hourly_v1  
  buffer: 10000

setup_precip_forc